In [ ]:
import pandas as pd
from pathlib import Path
# =========================
# 1. Load data
# =========================
# BASE_DIR = Path(__file__).resolve().parent
orders = pd.read_csv("/content/orders.csv")
payments = pd.read_csv("/content/payments.csv")
deliveries = pd.read_csv("/content/deliveries.csv")
# =========================
# 2. Basic inspection
# =========================
print("Orders shape:", orders.shape)
print("Payments shape:", payments.shape)
print("Deliveries shape:", deliveries.shape)

Orders shape: (240, 9)
Payments shape: (235, 6)
Deliveries shape: (213, 5)


In [ ]:
# =========================
# 3. Cleaning
# =========================
for df in [orders, payments, deliveries]:
    df.columns = df.columns.str.strip().str.lower()

orders["order_id"] = orders["order_id"].astype(str).str.strip()
payments["order_id"] = payments["order_id"].astype(str).str.strip()
deliveries["order_id"] = deliveries["order_id"].astype(str).str.strip()

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
payments["payment_date"] = pd.to_datetime(payments["payment_date"], errors="coerce")
deliveries["ship_date"] = pd.to_datetime(deliveries["ship_date"], errors="coerce")
deliveries["delivery_date"] = pd.to_datetime(deliveries["delivery_date"], errors="coerce")

In [ ]:
# =========================
# 4. Basic validation
# =========================
print("\nNull values:")
print("\nOrders:\n", orders.isnull().sum())
print("\nPayments:\n", payments.isnull().sum())
print("\nDeliveries:\n", deliveries.isnull().sum())

print("\nDuplicate order_id counts:")
print("Orders:", orders["order_id"].duplicated().sum())
print("Payments:", payments["order_id"].duplicated().sum())
print("Deliveries:", deliveries["order_id"].duplicated().sum())


Null values:

Orders:
 order_id         0
customer_id      0
order_date       0
amount           0
category         0
city             0
region           0
order_status     0
refund_status    0
dtype: int64

Payments:
 payment_id        0
order_id          0
payment_date      0
paid_amount       0
payment_method    0
payment_status    0
dtype: int64

Deliveries:
 delivery_id         0
order_id            0
ship_date          12
delivery_date      98
delivery_status     0
dtype: int64

Duplicate order_id counts:
Orders: 0
Payments: 0
Deliveries: 0


In [ ]:
# =========================
# 5. Set-based missing record check
# =========================
orders_set = set(orders["order_id"])
payments_set = set(payments["order_id"])
deliveries_set = set(deliveries["order_id"])

missing_in_payments = sorted(orders_set - payments_set)
missing_in_orders = sorted(payments_set - orders_set)
missing_in_deliveries = sorted(orders_set - deliveries_set)

print("\nOrders with no payment:", len(missing_in_payments))
print("Payments with no order:", len(missing_in_orders))
print("Orders with no delivery:", len(missing_in_deliveries))


Orders with no payment: 12
Payments with no order: 7
Orders with no delivery: 27


In [ ]:
# =========================
# 6. Merge orders + payments
# =========================
op = pd.merge(
    orders,
    payments,
    on="order_id",
    how="outer",
    indicator=True,
    suffixes=("_order", "_payment")
)

# =========================
# 7. Merge delivery data
# =========================
full_data = pd.merge(
    op,
    deliveries,
    on="order_id",
    how="outer"
)


In [ ]:
# =========================
# 8. Rule-based issue classification
# =========================
def classify_issue(row):
    order_exists = pd.notnull(row.get("customer_id"))
    payment_exists = pd.notnull(row.get("payment_id"))
    delivery_exists = pd.notnull(row.get("delivery_id"))

    order_amount = row.get("amount")
    paid_amount = row.get("paid_amount")
    order_status = str(row.get("order_status")).lower() if pd.notnull(row.get("order_status")) else ""
    payment_status = str(row.get("payment_status")).lower() if pd.notnull(row.get("payment_status")) else ""
    delivery_status = str(row.get("delivery_status")).lower() if pd.notnull(row.get("delivery_status")) else ""
    refund_status = str(row.get("refund_status")).lower() if pd.notnull(row.get("refund_status")) else ""

    if order_exists and not payment_exists:
        return "Order present, payment missing"
    if payment_exists and not order_exists:
        return "Payment present, order missing"
    if order_exists and payment_exists and pd.notnull(order_amount) and pd.notnull(paid_amount) and order_amount != paid_amount:
        return "Amount mismatch"
    if payment_status in ["failed", "pending"] and delivery_status == "delivered":
        return "Delivered but payment failed/pending"
    if payment_status == "success" and not delivery_exists and order_status != "cancelled":
        return "Payment success but delivery missing"
    if order_status == "cancelled" and refund_status == "pending":
        return "Cancelled order, refund pending"
    if order_status == "confirmed" and payment_status in ["failed", "pending"]:
        return "Order confirmed but payment not successful"
    return "Matched / No issue"

full_data["issue_type"] = full_data.apply(classify_issue, axis=1)

In [ ]:

import pandas as pd
from pathlib import Path

# =========================
# 1. Load data
# =========================
# BASE_DIR = Path(__file__).resolve().parent
orders = pd.read_csv("/content/orders.csv")
payments = pd.read_csv("/content/payments.csv")
deliveries = pd.read_csv("/content/deliveries.csv")

# =========================
# 2. Basic inspection
# =========================
print("Orders shape:", orders.shape)
print("Payments shape:", payments.shape)
print("Deliveries shape:", deliveries.shape)

# =========================
# 3. Cleaning
# =========================
for df in [orders, payments, deliveries]:
    df.columns = df.columns.str.strip().str.lower()

orders["order_id"] = orders["order_id"].astype(str).str.strip()
payments["order_id"] = payments["order_id"].astype(str).str.strip()
deliveries["order_id"] = deliveries["order_id"].astype(str).str.strip()

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
payments["payment_date"] = pd.to_datetime(payments["payment_date"], errors="coerce")
deliveries["ship_date"] = pd.to_datetime(deliveries["ship_date"], errors="coerce")
deliveries["delivery_date"] = pd.to_datetime(deliveries["delivery_date"], errors="coerce")

# =========================
# 4. Basic validation
# =========================
print("\nNull values:")
print("\nOrders:\n", orders.isnull().sum())
print("\nPayments:\n", payments.isnull().sum())
print("\nDeliveries:\n", deliveries.isnull().sum())

print("\nDuplicate order_id counts:")
print("Orders:", orders["order_id"].duplicated().sum())
print("Payments:", payments["order_id"].duplicated().sum())
print("Deliveries:", deliveries["order_id"].duplicated().sum())

# =========================
# 5. Set-based missing record check
# =========================
orders_set = set(orders["order_id"])
payments_set = set(payments["order_id"])
deliveries_set = set(deliveries["order_id"])

missing_in_payments = sorted(orders_set - payments_set)
missing_in_orders = sorted(payments_set - orders_set)
missing_in_deliveries = sorted(orders_set - deliveries_set)

print("\nOrders with no payment:", len(missing_in_payments))
print("Payments with no order:", len(missing_in_orders))
print("Orders with no delivery:", len(missing_in_deliveries))

# =========================
# 6. Merge orders + payments
# =========================
op = pd.merge(
    orders,
    payments,
    on="order_id",
    how="outer",
    indicator=True,
    suffixes=("_order", "_payment")
)

# =========================
# 7. Merge delivery data
# =========================
full_data = pd.merge(
    op,
    deliveries,
    on="order_id",
    how="outer"
)

# =========================
# 8. Rule-based issue classification
# =========================
def classify_issue(row):
    order_exists = pd.notnull(row.get("customer_id"))
    payment_exists = pd.notnull(row.get("payment_id"))
    delivery_exists = pd.notnull(row.get("delivery_id"))

    order_amount = row.get("amount")
    paid_amount = row.get("paid_amount")
    order_status = str(row.get("order_status")).lower() if pd.notnull(row.get("order_status")) else ""
    payment_status = str(row.get("payment_status")).lower() if pd.notnull(row.get("payment_status")) else ""
    delivery_status = str(row.get("delivery_status")).lower() if pd.notnull(row.get("delivery_status")) else ""
    refund_status = str(row.get("refund_status")).lower() if pd.notnull(row.get("refund_status")) else ""

    if order_exists and not payment_exists:
        return "Order present, payment missing"
    if payment_exists and not order_exists:
        return "Payment present, order missing"
    if order_exists and payment_exists and pd.notnull(order_amount) and pd.notnull(paid_amount) and order_amount != paid_amount:
        return "Amount mismatch"
    if payment_status in ["failed", "pending"] and delivery_status == "delivered":
        return "Delivered but payment failed/pending"
    if payment_status == "success" and not delivery_exists and order_status != "cancelled":
        return "Payment success but delivery missing"
    if order_status == "cancelled" and refund_status == "pending":
        return "Cancelled order, refund pending"
    if order_status == "confirmed" and payment_status in ["failed", "pending"]:
        return "Order confirmed but payment not successful"
    return "Matched / No issue"

full_data["issue_type"] = full_data.apply(classify_issue, axis=1)

# =========================
# 9. Recommended actions
# =========================
action_map = {
    "Order present, payment missing": "Check payment gateway logs and retry status sync.",
    "Payment present, order missing": "Check order creation pipeline or duplicate payment flow.",
    "Amount mismatch": "Verify fees, discounts, or partial payment logic.",
    "Delivered but payment failed/pending": "Urgent finance + operations review.",
    "Payment success but delivery missing": "Check fulfillment / OMS integration.",
    "Cancelled order, refund pending": "Escalate refund processing to finance.",
    "Order confirmed but payment not successful": "Verify payment status and customer notification.",
    "Matched / No issue": "No action required."
}
full_data["recommended_action"] = full_data["issue_type"].map(action_map)

# =========================
# 10. Business metrics
# =========================
total_orders = orders["order_id"].nunique()
total_gmv = orders["amount"].sum()

confirmed_mask = (
    full_data["payment_status"].eq("success") &
    full_data["issue_type"].eq("Matched / No issue")
)
confirmed_gmv = full_data.loc[confirmed_mask, "paid_amount"].sum()

amount_at_risk = full_data.loc[
    full_data["issue_type"] != "Matched / No issue", "amount"
].fillna(0).sum()

success_rate = (
    (payments["payment_status"] == "success").sum() / len(payments) * 100
    if len(payments) > 0 else 0
)

kpi_summary = pd.DataFrame({
    "metric": [
        "Total Orders",
        "Total GMV",
        "Confirmed GMV",
        "Amount At Risk",
        "Payment Success Rate (%)",
        "Total Issues Found"
    ],
    "value": [
        total_orders,
        total_gmv,
        confirmed_gmv,
        amount_at_risk,
        round(success_rate, 2),
        (full_data["issue_type"] != "Matched / No issue").sum()
    ]
})

# =========================
# 11. Breakdown tables
# =========================
daily_trend = (
    orders.groupby(orders["order_date"].dt.date)
    .agg(
        orders_count=("order_id", "count"),
        gmv=("amount", "sum")
    )
    .reset_index()
    .rename(columns={"order_date": "date"})
)

payment_method_summary = (
    full_data.groupby("payment_method", dropna=False)
    .agg(
        total_transactions=("order_id", "count"),
        total_paid_amount=("paid_amount", "sum")
    )
    .reset_index()
    .sort_values("total_transactions", ascending=False)
)

category_summary = (
    full_data.groupby("category", dropna=False)
    .agg(
        total_orders=("order_id", "count"),
        total_sales=("amount", "sum")
    )
    .reset_index()
    .sort_values("total_sales", ascending=False)
)

city_summary = (
    full_data.groupby(["region", "city"], dropna=False)
    .agg(
        total_orders=("order_id", "count"),
        total_sales=("amount", "sum")
    )
    .reset_index()
    .sort_values(["region", "total_sales"], ascending=[True, False])
)

issue_summary = (
    full_data.groupby("issue_type")
    .agg(
        issue_count=("order_id", "count"),
        amount_impact=("amount", "sum")
    )
    .reset_index()
    .sort_values("issue_count", ascending=False)
)

# =========================
# 12. Discrepancy report
# =========================
discrepancy_report = full_data.loc[
    full_data["issue_type"] != "Matched / No issue",
    [
        "order_id", "customer_id", "order_date", "amount", "category", "city", "region",
        "order_status", "refund_status", "payment_id", "payment_date", "paid_amount",
        "payment_method", "payment_status", "delivery_id", "ship_date", "delivery_date",
        "delivery_status", "issue_type", "recommended_action"
    ]
].sort_values(["issue_type", "order_id"])

# =========================
# 13. Export output reports
# =========================
BASE_DIR = Path("/content")
output_dir = BASE_DIR / "reports"
output_dir.mkdir(exist_ok=True)

full_data.to_csv(output_dir / "master_reconciliation_table.csv", index=False)
discrepancy_report.to_csv(output_dir / "discrepancy_report.csv", index=False)
daily_trend.to_csv(output_dir / "daily_trend.csv", index=False)
payment_method_summary.to_csv(output_dir / "payment_method_summary.csv", index=False)
category_summary.to_csv(output_dir / "category_summary.csv", index=False)
city_summary.to_csv(output_dir / "city_summary.csv", index=False)
issue_summary.to_csv(output_dir / "issue_summary.csv", index=False)
kpi_summary.to_csv(output_dir / "kpi_summary.csv", index=False)

print("\nDone. Reports saved inside:", output_dir)
print("\nTop issue summary:")
print(issue_summary.head(10))


Orders shape: (240, 9)
Payments shape: (235, 6)
Deliveries shape: (213, 5)

Null values:

Orders:
 order_id         0
customer_id      0
order_date       0
amount           0
category         0
city             0
region           0
order_status     0
refund_status    0
dtype: int64

Payments:
 payment_id        0
order_id          0
payment_date      0
paid_amount       0
payment_method    0
payment_status    0
dtype: int64

Deliveries:
 delivery_id         0
order_id            0
ship_date          12
delivery_date      98
delivery_status     0
dtype: int64

Duplicate order_id counts:
Orders: 0
Payments: 0
Deliveries: 0

Orders with no payment: 12
Payments with no order: 7
Orders with no delivery: 27

Done. Reports saved inside: /content/reports

Top issue summary:
                                   issue_type  issue_count  amount_impact
3                          Matched / No issue          198       247319.0
5              Order present, payment missing           12        19782.0
7